In [ ]:
!pip install -Uqqq scikit-learn

In [ ]:
import sklearn
sklearn.__version__

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score as acc

In [ ]:
from sklearn.tree import export_graphviz
import graphviz
import re

def draw_tree(t, df, size=10, ratio=0.6, precision=0, **kwargs):
    s=export_graphviz(t, out_file=None, feature_names=df.columns, filled=True, rounded=True,
                      special_characters=True, rotate=False, precision=precision, **kwargs)
    return graphviz.Source(re.sub('Tree {', f'Tree {{ size={size}; ratio={ratio}', s))

In [ ]:
train_df = pd.read_csv('../input/titanic/train.csv')
train_df.tail(2)

In [ ]:
test_df = pd.read_csv('../input/titanic/test.csv')
test_df.tail(2)

# Train/valid Split

In [ ]:
X = train_df.copy()
y = X.pop('Survived')

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

# Baselines

## Predicting always 0 (dead, the most common class)

In [ ]:
dummy_most_freq = DummyClassifier(strategy = 'most_frequent').fit(X_train, y_train)
y_pred = dummy_most_freq.predict(X_valid)
acc(y_pred, y_valid)

## Predicting all females survived

In [ ]:
pd.concat([X_train, y_train], axis = 1).groupby('Sex')['Survived'].mean()

In [ ]:
(X_train['Sex'] == 'female').mean()

In [ ]:
def dummy_female(X): return (X['Sex'] == 'female').astype(int)
y_pred = dummy_female(X_valid)
acc(y_pred, y_valid)

# Evaluating using CV

In [ ]:
X.head(1)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, ExtraTreesClassifier
from sklearn.linear_model import RidgeClassifier
from sklearn.model_selection import KFold, cross_val_score, cross_val_predict
from sklearn.metrics import confusion_matrix
from sklearn.tree import DecisionTreeClassifier
import numpy as np

In [ ]:
kf = KFold(n_splits=10, random_state=42, shuffle=True)

# Baseline

In [ ]:
score = []
for train_id, valid_id in kf.split(X, y):
    X_train = X.loc[train_id]
    X_valid = X.loc[valid_id]
    y_train = y.loc[train_id]
    y_valid = y.loc[valid_id]
    
    y_pred = dummy_female(X_valid)
    score.append(acc(y_pred, y_valid))
score = np.array(score)
score, score.mean(), score.std()

# ML models

In [ ]:
cat_feat = ['Sex', 'Ticket', 'Cabin', 'Embarked']
num_feat = ['Pclass', 'Age', 'SibSp', 'Parch']

In [ ]:
num_transform = SimpleImputer(strategy = 'constant', fill_value = -1)

cat_transform = Pipeline(
    steps = [
        ('inputer', SimpleImputer(strategy = 'constant', fill_value = '#NA#')),
        ('encoder', OrdinalEncoder(handle_unknown = 'use_encoded_value', unknown_value = -1))
    ]
)

preprocessor = ColumnTransformer(
    transformers = [
        ('num', num_transform, num_feat),
        ('cat', cat_transform, cat_feat)
    ]
)

# DT (all male dies)

In [ ]:
pipeline = Pipeline(
    steps = [
        ('pre', preprocessor),
        ('model', DecisionTreeClassifier(max_depth = 1, random_state=42))
    ]
)
score = cross_val_score(pipeline, X, y, cv = kf)
score, score.mean(), score.std()

In [ ]:
pipeline.fit(X, y);
draw_tree(pipeline['model'], X[num_feat+cat_feat], size = 7)

In [ ]:
for md in [1, 2, 3, 5, 10, 20]:
    pipeline = Pipeline(
        steps = [
            ('pre', preprocessor),
            ('model', DecisionTreeClassifier(max_depth = md, random_state=42))
        ]
    )
    score = cross_val_score(pipeline, X, y, cv = kf)
    print(f'max_depth: {md:>2} | {score.mean():.3f}+-{score.std():.3f}')

# RF

In [ ]:
for md in [1, 2, 3, 4, 5, 6, 7, 8]:
    pipeline = Pipeline(
        steps = [
            ('pre', preprocessor),
            ('model', RandomForestClassifier(max_depth = md, random_state=42))
        ]
    )
    score = cross_val_score(pipeline, X, y, cv = kf)
    print(f'max_depth: {md:>2} | {score.mean():.3f}+-{score.std():.3f}')

# Ridge

In [ ]:
for a in [0.01, 0.1, 1, 10, 100, 1000]:
    pipeline = Pipeline(
        steps = [
            ('pre', preprocessor),
            ('model', RidgeClassifier(alpha = a, random_state=42))
        ]
    )
    score = cross_val_score(pipeline, X, y, cv = kf)
    print(f'max_depth: {md:>2} | {score.mean():.3f}+-{score.std():.3f}')

# ET

In [ ]:
for md in [1, 2, 3, 4, 5, 6, 7, 8]:
    pipeline = Pipeline(
        steps = [
            ('pre', preprocessor),
            ('model', ExtraTreesClassifier(max_depth = md, random_state=42))
        ]
    )
    score = cross_val_score(pipeline, X, y, cv = kf)
    print(f'max_depth: {md:>2} | {score.mean():.3f}+-{score.std():.3f}')

# Ensembling

In [ ]:
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', VotingClassifier(estimators=[
        ('rf', RandomForestClassifier(max_depth = 6, n_estimators=200, random_state=42)),
        ('ridge', RidgeClassifier(alpha = 100, random_state = 42)),
        ('et', ExtraTreesClassifier(max_depth = 4, n_estimators=200, random_state=42)), 
    ]))
])

score = cross_val_score(pipeline, X, y, cv = kf)
score, score.mean(), score.std()

# Final model

## Best single model

In [ ]:
best_single_model = RandomForestClassifier(max_depth = 6, random_state=42)
best_single_pipeline = Pipeline(
    steps = [
        ('pre', preprocessor),
        ('model', best_single_model)
    ]
).fit(X, y)

In [ ]:
preds = best_single_pipeline.predict(X)
(preds == y).mean()

## Ensemble

In [ ]:
ensemble_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', VotingClassifier(estimators=[
        ('rf', RandomForestClassifier(max_depth = 6, n_estimators=200, random_state=42)),
        ('ridge', RidgeClassifier(alpha = 100, random_state = 42)),
        ('et', ExtraTreesClassifier(max_depth = 4, n_estimators=200, random_state=42)), 
    ]))
]).fit(X, y);

In [ ]:
preds = ensemble_pipeline.predict(X)
(preds == y).mean()

# Saving

In [ ]:
from joblib import dump, load

dump(best_single_pipeline, 'best_single.pipe')
dump(ensemble_pipeline, 'ensemble.pipe')

# Sanity

In [ ]:
pipe = load('best_single.pipe')
preds_loaded = pipe.predict(X)
(preds_loaded == y).mean()

In [ ]:
pipe = load('ensemble.pipe')
preds_loaded = pipe.predict(X)
(preds_loaded == y).mean()

# Inference

In [ ]:
test_df = pd.read_csv('../input/titanic/test.csv')
test_df.tail(2)

In [ ]:
pipeline = load('ensemble.pipe')
predictions = test_df.copy()
predictions['Survived'] = pipeline.predict(predictions)

In [ ]:
submission = predictions[['PassengerId', 'Survived']]
submission.to_csv('submission.csv', index = False)
submission